# Bellman-Ford Algorithm — Interactive Step-by-Step Notebook

This notebook walks through the Bellman-Ford shortest-path algorithm **one step at a time**.

**What you will do:**
1. Build the graph
2. Understand node positions (separate from the algorithm)
3. Step through the algorithm interactively in a full-screen visualizer
4. View all steps at once as a static grid
5. Reconstruct and display the final shortest path
6. Solve practice exercises

**Key difference from Dijkstra:** Bellman-Ford scans **all directed edges** in each iteration
instead of greedily picking the closest unvisited node. This makes it slower — O(V * E) vs
O((V+E) log V) — but it correctly handles **negative edge weights** and can detect **negative cycles**.

## 1. Setup

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
print("Libraries loaded.")

## 2. The Graph

This is a **directed** graph with 7 nodes and negative edge weights — a case where Dijkstra
fails but Bellman-Ford works correctly.

```
Directed edges:
  1->2  (w =  6)      1->3  (w =  5)      1->4  (w =  5)
  2->5  (w = -1)
  3->2  (w = -2)      3->5  (w =  1)
  4->3  (w = -2)      4->6  (w = -1)
  5->7  (w =  3)
  6->7  (w =  3)
```

Each key maps to a list of `(neighbor, weight)` pairs for its **outgoing** edges only.

In [ ]:
graph = {
    '1': [('2', 6), ('3', 5), ('4', 5)],
    '2': [('5', -1)],
    '3': [('2', -2), ('5', 1)],
    '4': [('3', -2), ('6', -1)],
    '5': [('7', 3)],
    '6': [('7', 3)],
    '7': [],
}
nodes = list(graph.keys())
edges = [(u, v, w) for u, nbrs in graph.items() for v, w in nbrs]
print("Nodes:", nodes)
print(f"Directed edges ({len(edges)} total):")
for u, v, w in edges:
    print(f"  {u} -> {v}  (w = {w})")

## 3. Node Positions — What Are They?

> **"What is `positions`? Why do I need it?"**

`positions` tells matplotlib where to draw each circle on the canvas.
It has **nothing to do** with the algorithm — Bellman-Ford only cares about edge weights.

```
Canvas layout:

  y=2   .   2   .   .   5   .
  y=1   1   .   3   .   .   7
  y=0   .   4   .   .   6   .
        x=0 1.5     3.0 4.5
```

In [ ]:
positions = {
    '1': (0.0, 1.0),
    '2': (1.5, 2.0),
    '3': (1.5, 1.0),
    '4': (1.5, 0.0),
    '5': (3.0, 2.0),
    '6': (3.0, 0.0),
    '7': (4.5, 1.0),
}

# Quick sanity-check: draw just the node positions with directed edges
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')
ax.set_title("Node positions (directed edges shown)", color='white')
ax.set_xlim(-0.5, 5.2); ax.set_ylim(-0.6, 2.6); ax.axis('off')

drawn_edges = set()
for u, nbrs in graph.items():
    for v, w in nbrs:
        x1, y1 = positions[u]; x2, y2 = positions[v]
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='-|>', color='#334155', lw=1.5,
                                   shrinkA=18, shrinkB=18, mutation_scale=12), zorder=1)
        mx, my = (x1+x2)/2, (y1+y2)/2
        col = '#ff6464' if w < 0 else '#64748b'
        ax.text(mx, my, str(w), color=col, fontsize=8, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='#0d1117', ec='none'))

for node, (x, y) in positions.items():
    ax.add_patch(plt.Circle((x, y), 0.22, color='#334155', ec='#00e5ff', lw=2, zorder=2))
    ax.text(x, y, node, ha='center', va='center', color='white',
            fontweight='bold', fontsize=12, zorder=3)

plt.tight_layout(); plt.show()
print("Positions only — no algorithm run yet. Red weights are negative.")

## 4. Bellman-Ford — Record Every Step

### How the algorithm works

1. Set `dist[source] = 0`, all other distances to infinity.
2. Repeat **V - 1** times (V = number of nodes):
   - For **every directed edge** (u, v, w):
     - If `dist[u] + w < dist[v]`, update `dist[v] = dist[u] + w` and set `prev[v] = u`.
3. Do one extra pass. If any distance still improves, a **negative cycle** exists.

### Why V - 1 iterations?

A shortest path with no negative cycles visits each node at most once, using at most V - 1 edges.
After k iterations, all shortest paths that use at most k edges are correctly computed.

### What to watch in this graph

The cheapest path from **1 to 7** is NOT the most direct one:
- Direct: 1->2->5->7 = 6 + (-1) + 3 = **8**
- Optimal: 1->4->3->2->5->7 = 5 + (-2) + (-2) + (-1) + 3 = **3**

Dijkstra would immediately pick node 2 (dist=6) over nodes 3 and 4, missing the better path
through the negative edges. Bellman-Ford finds it correctly.

In [ ]:
def bellman_ford_steps(graph, source):
    nodes = list(graph.keys())
    V     = len(nodes)
    dist  = {n: math.inf for n in nodes}
    prev  = {n: None     for n in nodes}
    dist[source] = 0
    all_edges = [(u, v, w) for u, nbrs in graph.items() for v, w in nbrs]
    steps = []

    def snap(current_edge=None, updated=None, message="", iteration=0):
        steps.append({
            'dist':         dist.copy(),
            'prev':         prev.copy(),
            'current_edge': current_edge,
            'updated':      updated or [],
            'message':      message,
            'iteration':    iteration,
        })

    snap(message=(f"Start: dist[{source}]=0, all others=inf. "
                  f"Will run up to {V-1} iterations over {len(all_edges)} directed edges."))

    for i in range(1, V):
        snap(message=f"Iteration {i}/{V-1}: examine all {len(all_edges)} directed edges.",
             iteration=i)
        iter_upd = []
        for u, v, w in all_edges:
            if dist[u] == math.inf:
                continue
            nd = dist[u] + w
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                iter_upd.append(v)
                snap(current_edge=(u, v), updated=[v],
                     message=f"Iter {i}: relax {u}->{v} (w={w}), dist[{v}] improved to {nd}",
                     iteration=i)
        if iter_upd:
            seen = list(dict.fromkeys(iter_upd))
            snap(updated=seen,
                 message=(f"Iter {i}/{V-1} done. Updated: "
                           + ", ".join(f"{v}={dist[v]}" for v in seen)),
                 iteration=i)
        else:
            snap(message=f"Iter {i}/{V-1}: no improvements — converged early.",
                 iteration=i)
            break

    neg_cycle = any(dist[u] != math.inf and dist[u] + w < dist[v]
                    for u, nbrs in graph.items() for v, w in nbrs)
    if neg_cycle:
        snap(message="Negative cycle detected! Shortest-path distances are undefined.")
    else:
        target = nodes[-1]
        path, cur = [], target
        while cur: path.append(cur); cur = prev[cur]
        path.reverse()
        snap(message=(f"Done! Shortest path {source}->{target}: "
                      f"{' -> '.join(path)}  (cost={dist[target]})"),
             updated=path)

    return steps, dist, prev


steps, dist, prev = bellman_ford_steps(graph, source='1')
print(f"Recorded {len(steps)} steps.")
for i, s in enumerate(steps):
    print(f"  Step {i:02d}: {s['message']}")

## 5. Frame Drawing Function

This function renders a single algorithm snapshot onto a matplotlib axes.
Edges are drawn as **directed arrows** since direction matters for Bellman-Ford.

Color legend:

| Color         | Meaning                              |
|---------------|--------------------------------------|
| Orange border | Distance was just improved this step |
| Cyan border   | Target node of the current edge      |
| Dark          | Not yet updated                      |
| Gold          | Final shortest path                  |
| Red weight    | Negative edge weight                 |

In [ ]:
def draw_frame(ax, snap, positions, graph, step_num, total):
    ax.clear()
    ax.set_facecolor('#0d1117')
    ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.8, 2.8); ax.axis('off')

    dist_s    = snap['dist']
    curr_e    = snap['current_edge']
    updated   = snap['updated']
    iteration = snap['iteration']
    is_done   = (step_num == total - 1)

    # ── Draw directed edges ───────────────────────────────────────
    for u, nbrs in graph.items():
        for v, w in nbrs:
            x1, y1 = positions[u]; x2, y2 = positions[v]
            on_path = is_done and u in updated and v in updated
            is_curr = curr_e and curr_e == (u, v)
            if on_path:
                color, lw = '#ffd700', 3.0
            elif is_curr:
                color, lw = '#ff6b35', 2.5
            else:
                color, lw = '#2a2a3e', 1.5
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                        arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                       shrinkA=20, shrinkB=20, mutation_scale=12),
                        zorder=1)
            mx, my = (x1+x2)/2, (y1+y2)/2
            wcol = '#ff6464' if w < 0 else '#64748b'
            ax.text(mx, my, str(w), color=wcol, fontsize=9, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.25', fc='#0d1117', ec='none'), zorder=2)

    # ── Draw nodes ────────────────────────────────────────────────
    for node, (x, y) in positions.items():
        if is_done and node in updated:
            fc, ec, ew = '#b45309', '#ffd700', 3
        elif node in updated:
            fc, ec, ew = '#1e293b', '#ff6b35', 3
        elif curr_e and node == curr_e[1]:
            fc, ec, ew = '#164e63', '#00e5ff', 3
        else:
            fc, ec, ew = '#1e293b', '#475569', 1.5

        ax.add_patch(plt.Circle((x, y), 0.25, color=fc, ec=ec, lw=ew, zorder=3))
        tc = '#000' if (is_done and node in updated) else '#fff'
        ax.text(x, y, node, ha='center', va='center', fontsize=13,
                fontweight='bold', color=tc, zorder=4)

        d = dist_s[node]
        d_str = str(d) if d != math.inf else 'inf'
        if is_done and node in updated:
            dc = '#ffd700'
        elif node in updated:
            dc = '#ff6b35'
        elif curr_e and node == curr_e[1]:
            dc = '#00e5ff'
        else:
            dc = '#94a3b8'
        ax.text(x, y+0.38, d_str, ha='center', va='bottom', fontsize=9, color=dc, zorder=4)

    # ── Step message ──────────────────────────────────────────────
    ax.text(2.25, -0.62, snap['message'], ha='center', va='center',
            fontsize=9, color='#e2e8f0', wrap=True,
            bbox=dict(boxstyle='round,pad=0.5', fc='#12121a', ec='#1e293b'))

    iter_str = f" | iter {iteration}" if iteration > 0 and not is_done else ""
    ax.set_title(f"Step {step_num}/{total-1}{iter_str}", color='#64748b', fontsize=10)

    # ── Legend ────────────────────────────────────────────────────
    legend_items = [
        mpatches.Patch(color='#1e293b', ec='#ff6b35', label='Just relaxed'),
        mpatches.Patch(color='#164e63', ec='#00e5ff', label='Edge target'),
        mpatches.Patch(color='#b45309', ec='#ffd700', label='Shortest path'),
    ]
    ax.legend(handles=legend_items, loc='upper right', fontsize=8,
              facecolor='#12121a', edgecolor='#2a2a3e', labelcolor='white')


# Preview: initial state and a mid-algorithm step
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0d1117')
draw_frame(axes[0], steps[0], positions, graph, 0, len(steps))
draw_frame(axes[1], steps[5], positions, graph, 5, len(steps))
plt.tight_layout()
plt.show()

## 6. Step-by-Step Interactive Walkthrough

Run the cell below to open a full-screen interactive window.

- Use **Next >>** and **<< Prev** buttons to move through the algorithm one step at a time
- The orange arrow and highlighted node show the relaxation happening right now
- Keyboard shortcuts: arrow keys, Space (next), Esc (close)

In [ ]:
import subprocess, sys, os

script = os.path.join(os.path.dirname(os.path.abspath('bellmanford_pygame.py')), 'bellmanford_pygame.py')
subprocess.Popen([sys.executable, script])
print("Pygame window launched.")
print("Controls: [Next >>] / [<< Prev] buttons   |   Arrow keys   |   Space = next   |   Esc = close")

## 7. All Steps at a Glance

This cell renders every algorithm step as a static grid.
Useful for reviewing all steps at once, printing, or studying offline.

In [ ]:
n    = len(steps)
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(13, rows * 3.8))
fig.patch.set_facecolor('#0a0a0f')
axes = axes.flatten()

for i, snap in enumerate(steps):
    draw_frame(axes[i], snap, positions, graph, i, n)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Bellman-Ford Algorithm — All Steps", color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Final Distance Table & Path Reconstruction

In [ ]:
def reconstruct_path(prev, source, target):
    path, cur = [], target
    while cur:
        path.append(cur); cur = prev[cur]
    path.reverse()
    return path if path and path[0] == source else []

print(f"{'Node':>6} | {'Shortest Dist':>14} | {'Via':>4} | Path from 1")
print("-" * 55)
for node in nodes:
    path = reconstruct_path(prev, '1', node)
    d    = dist[node]
    via  = prev[node] or '—'
    d_str = str(d) if d != math.inf else 'inf'
    print(f"{node:>6} | {d_str:>14} | {via:>4} | {' -> '.join(path)}")

## 9. Complexity

| Property | Bellman-Ford | Dijkstra (binary heap) |
|---|---|---|
| **Time** | O(V * E) | O((V + E) log V) |
| **Space** | O(V) | O(V + E) |
| **Negative weights** | Yes | No |
| **Negative cycle detection** | Yes | No |
| **Typical performance** | Slower | Faster |

**V** = vertices (7 here), **E** = edges (10 here). So this run does at most 7 * 10 = 70 edge checks.

**When to use Bellman-Ford:**
- The graph contains negative edge weights
- You need to detect negative cycles
- Correctness matters more than raw speed

**Early termination:** If an entire iteration produces no improvements, the algorithm has converged
and stops. In this graph that happens after 3 iterations instead of the full 6.

## 10. Practice Exercises

### A — Different source
Run Bellman-Ford from node `'4'` as the source. What is the shortest path from 4 to 7?

In [ ]:
steps_a, dist_a, prev_a = bellman_ford_steps(graph, source='4')
path_a = reconstruct_path(prev_a, '4', '7')
print(f"4 -> 7: cost={dist_a['7']}, path={' -> '.join(path_a)}")

n_a  = len(steps_a)
cols = 3
rows = math.ceil(n_a / cols)

fig, axes = plt.subplots(rows, cols, figsize=(13, rows * 3.8))
fig.patch.set_facecolor('#0a0a0f')
axes = axes.flatten()

for i, snap in enumerate(steps_a):
    draw_frame(axes[i], snap, positions, graph, i, n_a)

for j in range(n_a, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Exercise A: Bellman-Ford from node 4", color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### B — Negative cycle detection

Add the edge **5 -> 3** with weight **-4**. This creates a cycle:

```
  3 -> 5  (w =  1)
  5 -> 3  (w = -4)
  Cycle cost: 1 + (-4) = -3  <  0  =>  negative cycle!
```

Bellman-Ford detects this in the extra (V-th) pass and reports it.

In [ ]:
import copy

graph_b = copy.deepcopy(graph)
graph_b['5'].append(('3', -4))  # creates negative cycle 3->5->3 = 1 + (-4) = -3

steps_b, dist_b, prev_b = bellman_ford_steps(graph_b, '1')
print("Last message:", steps_b[-1]['message'])
print()
print("Distances when cycle detected (unreliable — kept decreasing):")
for node in nodes:
    d = dist_b[node]
    print(f"  dist[{node}] = {d if d != math.inf else 'inf'}")

### C — Count relaxations per iteration

How many edge relaxations (actual distance improvements) occur in each iteration?

In [ ]:
total_relax = 0
by_iter = {}
for s in steps:
    if s['current_edge'] is not None and len(s['updated']) == 1:
        i = s['iteration']
        by_iter[i] = by_iter.get(i, 0) + 1
        total_relax += 1

print(f"Total relaxations: {total_relax}")
for i in sorted(by_iter):
    print(f"  Iteration {i}: {by_iter[i]} relaxation(s)")